# 라이브러리

In [44]:
# 아파트 매매 데이터 및 위경도 라이브러리

import pandas as pd
import numpy as np
import requests
from tqdm import tqdm
from dotenv import load_dotenv
import os
import glob
from pathlib import Path
from functools import reduce



In [45]:
# 위성지도 다운로드 라이브러리

# === 필요한 라이브러리 불러오기 ===
import ee  # Google Earth Engine 파이썬 API
import requests  # 웹 요청 (썸네일 이미지 다운로드)
import pandas as pd
from PIL import Image  # 이미지 파일 열고 저장
from io import BytesIO  # 이미지 바이트 데이터를 PIL로 읽기 위한 버퍼
from datetime import datetime, timedelta, date  # 날짜 처리용
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from dotenv import load_dotenv
import os 

import logging
from logging.handlers import RotatingFileHandler

# 위성지도 전처리 라이브러리
import os
from PIL import Image
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torchvision.transforms import ToPILImage

from tqdm import tqdm

In [139]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
import joblib

import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tcn import TCN

# 한글 폰트 설정
try:
    plt.rc('font', family='AppleGothic')
except:
    plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False


# 데이터 전처리

## 아파트

### 매매 데이터 불러오기

In [47]:
load_dotenv()
api_key = os.getenv("KAKAO_API_KEY")
print(api_key)

531d049fba15b8acbb290989f6988d89


In [48]:

# 위경도 추가된 아파트 매매 데이터가 저장되는 경로
OUTPUT_PATH = Path('../../data/interim/apt/validation_apt_with_long_lat.csv')  


In [49]:

# 사용할 컬럼만 지정 (NO 컬럼 제외)
columns_to_use = [
    '시군구', '단지명', '전용면적(㎡)', '계약년월', '계약일',
    '거래금액(만원)', '동', '층', '건축년도', '도로명'
]

# 병합할 CSV 파일이 들어있는 폴더 경로
folder_path = "../../data/raw/apt_sale/validation_apt_data.csv"
gangnam_path = "../../data/raw/apt_sale/gangnam"
df = pd.read_csv(
    folder_path,
    encoding='cp949',
    skiprows=15,               # 메타데이터 줄 건너뛰기
    usecols=columns_to_use     # 필요한 컬럼만 불러오기
)

# 병합된 결과 출력
print(f"\n 실제 데이터 불러오기 완료: 총 {len(df)}건")
min_contract = df['계약년월'].min()
max_contract = df['계약년월'].max()
print(f"데이터는 {min_contract} ~ {max_contract}까지의 거래로 이루어져 있습니다.")


 실제 데이터 불러오기 완료: 총 346건
데이터는 202507 ~ 202507까지의 거래로 이루어져 있습니다.


### 면적당 단가, 아파트 나이, 거래일자 순으로 나열 및 필요없는 컬럼 삭제

In [50]:
df['거래금액(만원)'] = df['거래금액(만원)'].str.replace(',', '').astype(int)
df['면적당 단가(만원)'] = df['거래금액(만원)'] / df['전용면적(㎡)']
df['계약년도'] = df['계약년월'].astype(str).str[:4].astype(int)
df['아파트 나이'] = df['계약년도'] - df['건축년도']
# 계약연-월-일을 기준으로 시계열 정렬
df['계약일자'] = df['계약년월'].astype(str) + df['계약일'].astype(str).str.zfill(2)
df['계약일자'] = pd.to_datetime(df['계약일자'], format='%Y%m%d')

df = df.sort_values('계약일자').reset_index(drop=True)
df.drop(['시군구','계약년월','계약일','동','계약년도','거래금액(만원)'], axis=1, inplace=True)

# 빠르게 하기 위해서 상위 100개만 예측에 사용해봄
df = df.head(100)
df.head(1)

,단지명,전용면적(㎡),층,건축년도,도로명,면적당 단가(만원),아파트 나이,계약일자
0,금호어울림,84.95,4,2003,도곡로7길 22,2295.467922,22,2025-07-01


### 위경도 수집

In [51]:

# === 좌표 변환 === #
headers = {'Authorization': f'KakaoAK {api_key}'}
#Authorization: KakaoAK ${REST_API_KEY}"
def get_coords(address):
    res = requests.get(
        "https://dapi.kakao.com/v2/local/search/address.json",
        headers=headers,
        params={'query': address}
    )
    if res.status_code == 200 and res.json()['documents']:
        doc = res.json()['documents'][0]
        return doc['x'], doc['y']
    return None, None

longitudes, latitudes = [], []
for address in tqdm(df['도로명'], desc="좌표 변환 중"):
    x, y = get_coords(address)
    longitudes.append(x)
    latitudes.append(y)

df['경도'] = longitudes
df['위도'] = latitudes

# === 최종 정제 및 저장 === #

df['면적당 단가(만원)'] = np.log(df['면적당 단가(만원)'])

# === 수집 못한 위도 경도는 삭제 === #
df.dropna(inplace=True)

# 디렉토리 없으면 생성
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f"✅ 저장 완료: {OUTPUT_PATH}")

좌표 변환 중: 100%|██████████| 100/100 [00:06<00:00, 15.44it/s]

✅ 저장 완료: ../../data/interim/apt/validation_apt_with_long_lat.csv


# 위성지도

## 위성지도 수집

In [52]:

def get_satellite_image():    


    # === 환경 변수 로드 ===
    load_dotenv()
    project = os.getenv("PROJECT")

    # === 경로 설정 ===
    DATA_PATH = '../../data/interim/apt/validation_apt_with_long_lat.csv'
    SAVE_PATH = '../../data/raw/validation_apt_images/'
    # 밑에껀 10년치 데이터 쓸때 사용하는 경로임

    # DATA_PATH = 'data/interim/apt/new/gang_nam_apt_with_long_lar.csv'
    # SAVE_PATH = 'data/raw/new/gang_nam_apt_images/'
    #LOG_PATH = "data/log/image_download/image_download.log"

    # === 경로 보장 ===
    #os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)
    os.makedirs(SAVE_PATH, exist_ok=True)

    # # === 로거 설정 ===
    # LOG_FORMAT = "[%(asctime)s] [%(levelname)s] %(message)s"
    # logger = logging.getLogger()
    # logger.setLevel(logging.INFO)

    # formatter = logging.Formatter(LOG_FORMAT)
    # file_handler = RotatingFileHandler(LOG_PATH, maxBytes=5*1024*1024, backupCount=5)
    # file_handler.setFormatter(formatter)
    # logger.addHandler(file_handler)

    # 콘솔 출력도 원하면 주석 해제
    # stream_handler = logging.StreamHandler()
    # stream_handler.setFormatter(formatter)
    # logger.addHandler(stream_handler)

    # === 병렬처리 설정 ===
    MAX_WORKERS = 1

    # === 데이터 불러오기 ===
    df = pd.read_csv(DATA_PATH)
    # 갑자기 왜 DB에서 가져오나?
    #df = fetch_contract_dat_lon_lat()
    #df = pd.DataFrame(df)
    df.dropna(inplace=True)
    len(f"df의 길이 {df}")
    print(df.head())
    print(df.info())
    #df = df.head(1000) # 테스트용

    # === Earth Engine 초기화 ===
    ee.Authenticate()
    ee.Initialize(project=project)
    print("Google Earth Engine 초기화 완료")

    # === 거래 데이터 리스트화 ===
    apt_transactions = df[['위도', '경도', '계약일자']].apply(
        lambda row: {
            'lat': row['위도'],
            'lon': row['경도'],
            'date': row['계약일자']
        }, axis=1
    ).tolist()
    print("아파트 거래 데이터 정의 완료")

    def process_transaction(idx, tx):
        try:
            lat = tx['lat']
            lon = tx['lon']

            tx_date_raw = tx['date']
            if isinstance(tx_date_raw, date):
                tx_date = datetime.combine(tx_date_raw, datetime.min.time())
            else:
                tx_date = datetime.strptime(tx_date_raw, "%Y-%m-%d")
            # tx_date = datetime.strptime(tx['date'], "%Y-%m-%d")
            start_date = (tx_date - timedelta(days=183)).strftime('%Y-%m-%d')
            end_date = (tx_date + timedelta(days=183)).strftime('%Y-%m-%d')

            center = ee.Geometry.Point([lon, lat])
            roi = center.buffer(1500).bounds()

            collection = (
                ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                .filterBounds(center)
                .filterDate(start_date, end_date)
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))
            )

            count = collection.size().getInfo()
            if count == 0:
                return f"[X] 이미지 없음 (index {idx}): 날짜={tx['date']}"

            image = (
                collection
                .sort('system:time_start', False)
                .first()
            )

            stats = image.reduceRegion(
                reducer=ee.Reducer.percentile([2, 98]),
                geometry=roi,
                scale=10,
                maxPixels=1e8
            ).getInfo()

            if not stats:
                return f"[X] 통계 없음 (index {idx}): 날짜={tx['date']}"

            b4_min = stats.get('B4_p2', 500)
            b4_max = stats.get('B4_p98', 3500)
            b3_min = stats.get('B3_p2', 500)
            b3_max = stats.get('B3_p98', 3500)
            b2_min = stats.get('B2_p2', 500)
            b2_max = stats.get('B2_p98', 3500)

            url = image.getThumbURL({
                'region': roi,
                'format': 'jpg',
                'bands': ['B4', 'B3', 'B2'],
                'min': [b4_min, b3_min, b2_min],
                'max': [b4_max, b3_max, b2_max],
                'scale': 10
            })

            if not url or not url.startswith("https://"):
                return f"[X] URL 생성 실패 (index {idx})"

            response = requests.get(url)
            if response.status_code != 200:
                return f"[X] 이미지 요청 실패 (index {idx}): 상태코드 {response.status_code}"

            img = Image.open(BytesIO(response.content))
            img.save(f"{SAVE_PATH}apt_image_{idx}.jpg")
            return  # 성공 시 출력 안함

        except Exception as e:
            return f"[X] 예외 발생 (index {idx}): {e}"

    # === 병렬 처리 실행 ===
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_transaction, idx, tx) for idx, tx in enumerate(apt_transactions)]
        for future in tqdm(as_completed(futures), total=len(futures)):
            result = future.result()
            if result:
                print(result)

In [53]:
get_satellite_image()

     단지명  전용면적(㎡)   층  건축년도         도로명  면적당 단가(만원)  아파트 나이        계약일자  \
0  금호어울림    84.95   4  2003    도곡로7길 22    7.738692      22  2025-07-01   
1  PH129   273.96  12  2020  압구정로79길 88    8.844382       5  2025-07-01   
2    한양7   110.25   1  1981  압구정로61길 37    8.477882      44  2025-07-01   
3   목련타운    99.79   8  1993   광평로19길 15    7.903109      32  2025-07-01   
4    신동아    49.96  11  1992   광평로47길 17    8.071706      33  2025-07-01   

           경도         위도  
0  127.034771  37.492374  
1  127.054237  37.525956  
2  127.043861  37.529584  
3  127.084798  37.485531  
4  127.098186  37.487909  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   단지명         98 non-null     object 
 1   전용면적(㎡)     98 non-null     float64
 2   층           98 non-null     int64  
 3   건축년도        98 non-null     int64  
 4   도로명         98 non-null     objec

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_7TDKVSyKvBdmMqW?ref=4i2o6


Google Earth Engine 초기화 완료
아파트 거래 데이터 정의 완료


100%|██████████| 98/98 [06:43<00:00,  4.11s/it]


## 위성지도 전처리

In [54]:

def prepro_satellite():

    input_folder = '../../data/raw/validation_apt_images/'
    output_folder = '../../data/interim/satellites/validation_satellites'
    # 다양한 각도로 Random Rotation하여 실험해보기
    degrees = [0, 90, 180, 270, 360]

    base_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(
            # mean=[0.485, 0.456, 0.406],
            # std=[0.229, 0.224, 0.225]
            mean=[0.0, 0.0, 0.0],  # 평균 0
            std=[1.0, 1.0, 1.0]    # 분산 1
        )
    ])


    os.makedirs(output_folder, exist_ok=True)


    to_pil = ToPILImage()

    for degree in degrees:
        degree_folder = os.path.join(output_folder, f'rot_{degree}')
        os.makedirs(degree_folder, exist_ok=True)

        filenames = [f for f in os.listdir(input_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        for filename in tqdm(filenames, desc=f'Rotation {degree}'):
            img_path = os.path.join(input_folder, filename)
            img = Image.open(img_path).convert('RGB')

            rotated_img = TF.rotate(img, degree, fill=(255, 255, 255))

            augmented_img = base_transform(rotated_img)
            augmented_img_pil = to_pil(augmented_img)

            save_path = os.path.join(degree_folder, filename)
            augmented_img_pil.save(save_path)

prepro_satellite()

Rotation 360: 100%|██████████| 98/98 [00:00<00:00, 343.65it/s]


# 경제 지표 데이터

## 데이터 불러오기

In [96]:
# 경기종합지수
ECONOMIC_INDEX = '../../data/raw/validation_economies/경기종합지수.csv'
# 공종별 건설기성액
CONSTRUCTION_WORK = '../../data/raw/validation_economies/공종별_건설기성액.csv'
# 부동산시장 소비심리지수
REAL_ESTATE_CONSUMER_INDEX = '../../data/raw/validation_economies/부동산시장_소비심리지수.csv'
# 주택시장 소비심리지수
HOUSING_CONSUMER_SENTIMENT = '../../data/raw/validation_economies/주택시장_소비심리지수.csv'
# 지역별 지가변동률
PRICE_CHANGE_RATE = '../../data/raw/validation_economies/지역별_지가변동률.csv'
# 토지시장 소비심리지수
LAND_MARKET_CONSUMER_SENTIMENT_INDEX = '../../data/raw/validation_economies/토지시장_소비심리지수.csv'
# 행정구역별_아파트매매거래현황
APT_TRANSACTIONS = '../../data/raw/validation_economies/행정구역별_아파트매매거래현황.csv'
# 금리데이터
INTEREST_RATE = '../../data/raw/validation_economies/한국은행_기준금리.csv'


In [ ]:
#경기종합지수
EI = pd.read_csv(ECONOMIC_INDEX, encoding='cp949')
#공종별 건설기성액
CW = pd.read_csv(CONSTRUCTION_WORK, encoding='cp949')
#부동산시장 소비심리지수
RECI = pd.read_csv(REAL_ESTATE_CONSUMER_INDEX, encoding='cp949')
#주택시장 소비심리지수
HCS = pd.read_csv(HOUSING_CONSUMER_SENTIMENT, encoding='cp949')
#지역별 지가변동률
PCR = pd.read_csv(PRICE_CHANGE_RATE, encoding='cp949')
#토지시장 소비심리지수
LMCSI = pd.read_csv(LAND_MARKET_CONSUMER_SENTIMENT_INDEX, encoding='cp949')
#행정구역별 아파트 매매거래현황
AT = pd.read_csv(APT_TRANSACTIONS, encoding='cp949')
# 금리
IR = pd.read_csv(INTEREST_RATE)

### 각 데이터 별로 전처리 후 병합하기

In [ ]:
# 선행종합지수
EI = EI.iloc[2:,1:]
# 자료시점 컬럼을 datetime으로 변환
EI["자료시점"] = pd.to_datetime(EI["자료시점"], format="%Y년 %m월")
EI["자료시점"] = EI["자료시점"].dt.strftime("%Y-%m")
EI["선행종합지수"]=EI['선행종합지수'].astype(float)
EI = EI.reset_index(drop=True)
EI = EI.rename(columns={"자료시점":"month","선행종합지수":"leading_index"})

# 지역별 지가변동률
PCR.rename(columns={"자료시점":"month","서울":"강남구_변동률","서울.1":"강남구_누계"}, inplace=True)
PCR = PCR.iloc[3:, 1:]
PCR['month'] = pd.to_datetime(PCR['month'], format="%Y년 %m월")
PCR['month'] = PCR['month'].dt.strftime("%Y-%m")
PCR[['강남구_변동률','강남구_누계']] = PCR[['강남구_변동률','강남구_누계']].astype(float)

# 부동산시장 소비심리지수
RECI.rename(columns={"자료시점":"month","수도권":"부동산_소비심리지수"},inplace=True)
RECI = RECI.iloc[3:,1:]
RECI['month'] = pd.to_datetime(RECI['month'], format='%Y년 %m월')
RECI['month'] = RECI['month'].dt.strftime("%Y-%m")
RECI['부동산_소비심리지수'] = RECI['부동산_소비심리지수'].astype(float)

# 주택시장 소비심리지수
HCS.rename(columns={"자료시점":"month","수도권":"주택시장_소비심리지수"},inplace=True)
HCS = HCS.iloc[3:, 1:]
HCS['month'] = pd.to_datetime(HCS['month'], format="%Y년 %m월")
HCS['month'] = HCS['month'].dt.strftime("%Y-%m")
HCS["주택시장_소비심리지수"] = HCS['주택시장_소비심리지수'].astype(float)

# 토지시장 소비심리지수
LMCSI.rename(columns={'자료시점':'month','수도권':'토지시장_소비심리지수'},inplace=True)
LMCSI = LMCSI.iloc[3:,1:]
LMCSI['month'] = pd.to_datetime(LMCSI['month'], format="%Y년 %m월")
LMCSI['month'] = LMCSI['month'].dt.strftime("%Y-%m")
LMCSI['토지시장_소비심리지수'] = LMCSI['토지시장_소비심리지수'].astype(float)

# 아파트 매매 거래 현황
AT.rename(columns={'자료시점':'month','서울':'아파트_호수','서울.1':'아파트_면적'},inplace=True)
AT = AT.iloc[3:,1:]
AT['month'] = pd.to_datetime(AT['month'], format="%Y년 %m월")
AT['month'] = AT['month'].dt.strftime("%Y-%m")
AT[['아파트_호수','아파트_면적']] = AT[['아파트_호수','아파트_면적']].astype(float)

# 공종별 건설기성액
CW.rename(columns={"자료시점":"month",'계':"건설기성액(백만원)"},inplace=True)
CW = CW.iloc[2:,1:]
CW['month'] = pd.to_datetime(CW['month'], format="%Y년 %m월")
CW['month'] = CW['month'].dt.strftime("%Y-%m")
CW['건설기성액(백만원)'] = CW['건설기성액(백만원)']\
    .str.replace('"', '')\
    .str.replace(',', '')\
    .astype(float)

# 금리

IR.rename(columns={'변환':'month', '원자료':'rate'}, inplace=True)
# datetime 변환
IR['month'] = pd.to_datetime(IR['month'], format='%Y/%m')
IR['month'] = IR['month'].dt.strftime('%Y-%m')

In [116]:
sale = pd.read_csv('../../data/final/validation/validation_img_features.csv')

# datetime 변환
sale['계약일자'] = pd.to_datetime(sale['계약일자'], errors='coerce')  # 변환 불가 값은 NaT 처리
# 월 단위 추출
sale['month'] = sale['계약일자'].dt.strftime("%Y-%m")
drop_col = ['단지명','도로명','경도','위도']
sale.drop(drop_col, axis=1, inplace=True)

# 모든 경제지표 데이터와 병합후 저장하기
dfs = [sale, EI, CW, RECI, HCS, PCR, LMCSI, AT, IR]
merged_df = reduce(lambda left, right: pd.merge(left, right, on='month', how='left'), dfs)
merged_df.head()

merged_df.to_csv('../../data/interim/validation_final_numeric_data.csv',index=False)

In [117]:
merged_df.head(1)

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,계약일자,img_pca_0,img_pca_1,img_pca_2,img_pca_3,...,leading_index,건설기성액(백만원),부동산_소비심리지수,주택시장_소비심리지수,강남구_변동률,강남구_누계,토지시장_소비심리지수,아파트_호수,아파트_면적,rate
0,84.95,4,2003,7.738692,22,2025-07-01,-3.984524,9.133594,-10.304125,-3.682686,...,120.5,11428695.0,108.6,110.8,0.552,3.374,89.0,604.0,54.0,2.5


# TCN 모델로 예측

In [169]:
# ===================================================================
# PART 1: 기본 설정 및 저장된 자산(모델, 스케일러) 불러오기
# ===================================================================
print("PART 1: 저장된 모델과 스케일러를 불러옵니다...")

# --- 경로 및 파라미터 ---
TABULAR_DATA_PATH = '../../data/interim/validation_final_numeric_data.csv'
TIMESTEPS = 36
FORECAST_HORIZON = 12

# --- 저장된 파일 로드 ---
custom_objects = {'TCN': TCN}
final_model = load_model('../2week/final_tcn_model.h5', custom_objects=custom_objects)
scaler_X = joblib.load('../2week/scaler_X.pkl')

print("PART 1: 완료!")


PART 1: 저장된 모델과 스케일러를 불러옵니다...
PART 1: 완료!


In [170]:
# ===================================================================
# PART 2: 데이터 로딩 및 수동 시나리오 설정
# ===================================================================
print("\nPART 2: 전체 데이터를 로딩하고 시나리오를 설정합니다...")
df = pd.read_csv(TABULAR_DATA_PATH)
df['계약일자'] = pd.to_datetime(df['계약일자'])
df = df.sort_values('계약일자').reset_index(drop=True)

# --- 특징 컬럼 정의 ---
image_feature_cols = [col for col in df.columns if 'img_pca_' in col]
tabular_cols_to_use = [
    '전용면적(㎡)', '층', '건축년도', '아파트 나이', 'leading_index', '건설기성액(백만원)', 
    '부동산_소비심리지수', '주택시장_소비심리지수', '강남구_변동률', '강남구_누계', 
    '토지시장_소비심리지수', '아파트_호수', '아파트_면적', 'rate'
]
final_feature_cols = tabular_cols_to_use + image_feature_cols
TARGET_COL = '면적당 단가(만원)'
X = df[final_feature_cols]

# --- 수동 시나리오 정의 ---
economic_indicators_for_scenario = [
    'leading_index', '건설기성액(백만원)', '부동산_소비심리지수', '주택시장_소비심리지수', 
    '강남구_변동률', '강남구_누계', '토지시장_소비심리지수', 'rate'
]
scenarios = {
    '낙관적': { 'leading_index': 1.002, '건설기성액(백만원)': 1.003, '부동산_소비심리지수': 1.004, '주택시장_소비심리지수': 1.004, '강남구_변동률': 1.005, '강남구_누계': 1.003, '토지시장_소비심리지수': 1.003, 'rate': 0.999 },
    '중립적': { 'leading_index': 1.0, '건설기성액(백만원)': 1.0, '부동산_소비심리지수': 1.0, '주택시장_소비심리지수': 1.0, '강남구_변동률': 1.0, '강남구_누계': 1.0, '토지시장_소비심리지수': 1.0, 'rate': 1.0 },
    '보수적': { 'leading_index': 0.998, '건설기성액(백만원)': 0.997, '부동산_소비심리지수': 0.996, '주택시장_소비심리지수': 0.996, '강남구_변동률': 0.995, '강남구_누계': 0.997, '토지시장_소비심리지수': 0.997, 'rate': 1.001 }
}
print("수동으로 설정된 시나리오를 사용합니다.")
print("PART 2: 완료!")



PART 2: 전체 데이터를 로딩하고 시나리오를 설정합니다...
수동으로 설정된 시나리오를 사용합니다.
PART 2: 완료!


In [171]:

# ===================================================================
# PART 3: 시나리오 기반 미래 예측 수행
# ===================================================================
print("\nPART 3: 시나리오 기반 미래 예측을 수행합니다...")

if len(X) < TIMESTEPS:
    raise ValueError(f"예측을 위해 최소 {TIMESTEPS}개의 데이터가 필요하지만, 현재 데이터는 {len(X)}개입니다.")
last_sequence_data = X.tail(TIMESTEPS).values
scenario_log_predictions = {name: [] for name in scenarios.keys()} # ❗️ 로그 값 저장을 위해 이름 변경
current_inputs = {name: last_sequence_data.copy() for name in scenarios.keys()}

for i in tqdm(range(FORECAST_HORIZON), desc="미래 예측 중"):
    for name, params in scenarios.items():
        current_inputs_df = pd.DataFrame(current_inputs[name], columns=final_feature_cols)
        current_input_scaled = scaler_X.transform(current_inputs_df)
        current_input_reshaped = np.expand_dims(current_input_scaled, axis=0)
        
        # 모델의 예측 결과 (로그 스케일 값)
        prediction_log = final_model.predict(current_input_reshaped, verbose=0)
        
        # ❗️ 역변환 없이 로그 스케일 값 그대로 저장
        scenario_log_predictions[name].append(prediction_log[0][0])
        
        # 다음 스텝의 입력 데이터 생성
        next_feature_row = current_inputs[name][-1, :].copy()
        for indicator in economic_indicators_for_scenario:
            col_idx = X.columns.get_loc(indicator)
            next_feature_row[col_idx] *= params[indicator]
        updated_sequence = np.vstack([current_inputs[name][1:], next_feature_row])
        current_inputs[name] = updated_sequence
print("PART 3: 완료!")



PART 3: 시나리오 기반 미래 예측을 수행합니다...


미래 예측 중: 100%|██████████| 12/12 [00:00<00:00, 19.95it/s]

PART 3: 완료!


In [173]:
# ===================================================================
# PART 4: 시나리오 예측 결과 수치로 출력 (역변환 적용)
# ===================================================================
import joblib # joblib import 추가
import numpy as np # numpy import 추가

print("\nPART 4: 시나리오 예측 결과를 수치로 출력합니다...")

# [추가] Target 변수용 스케일러를 불러옵니다. (파일명은 실제 파일에 맞게 수정하세요)
try:
    scaler_y = joblib.load('../2week/scaler_y.pkl') 
except FileNotFoundError:
    print("🚨 'scaler_y.pkl' 파일을 찾을 수 없습니다. Target 스케일러 파일 경로를 확인해주세요.")
    # 파일이 없는 경우, 임시로 스크립트를 중단하거나 대체 로직을 구현할 수 있습니다.
    exit()


last_known_date = df['계약일자'].max()
future_dates = pd.date_range(start=last_known_date + pd.DateOffset(months=1), periods=FORECAST_HORIZON, freq='MS')

# 최종 예측 결과를 담을 데이터프레임 생성
original_scale_predictions_df = pd.DataFrame(index=future_dates)
original_scale_predictions_df.index.name = '예측 시점'

# [수정] 각 시나리오별로 올바른 역변환(스케일링 -> 로그) 수행
for name in scenarios.keys():
    # 1. 예측된 로그 값을 2D 배열로 변환
    log_preds = np.array(scenario_log_predictions[name]).reshape(-1, 1)
    
    # 2. 스케일링 역변환을 먼저 수행
    unscaled_log_preds = scaler_y.inverse_transform(log_preds)
    
    # 3. 그 다음 로그 역변환(np.expm1)을 수행하여 최종 가격 예측
    final_preds = np.expm1(unscaled_log_preds)
    
    # 4. 데이터프레임에 저장
    original_scale_predictions_df[f'{name} 예측(만원)'] = final_preds

pd.options.display.float_format = '{:.2f}'.format
print("\n" + "="*55)
print("          시나리오 기반 미래 12개월 매매 가격 예측 (실제 단위)")
print("="*55)
print(original_scale_predictions_df)
print("="*55)

print("\n🎉 모든 과정이 성공적으로 완료되었습니다!")


PART 4: 시나리오 예측 결과를 수치로 출력합니다...

          시나리오 기반 미래 12개월 매매 가격 예측 (실제 단위)
            낙관적 예측(만원)  중립적 예측(만원)  보수적 예측(만원)
예측 시점                                         
2025-09-01     2744.27     2744.27     2744.27
2025-10-01     2836.29     2832.73     2829.16
2025-11-01     2516.58     2508.44     2500.33
2025-12-01     2724.41     2712.93     2703.62
2026-01-01     2672.31     2653.16     2634.69
2026-02-01     2686.12     2665.46     2645.75
2026-03-01     2736.54     2700.96     2667.55
2026-04-01     2707.16     2664.61     2622.54
2026-05-01     2546.67     2503.80     2463.96
2026-06-01     2795.22     2739.17     2685.91
2026-07-01     2735.62     2659.52     2592.77
2026-08-01     2720.34     2631.20     2553.08

🎉 모든 과정이 성공적으로 완료되었습니다!
